# 15. Összefoglaló Kutatási Vezérlőpult és Tudományos Szintézis
## Executive Master Dashboard & Policy Synthesis

**Cél**: A teljes 16 modulos kőbányai ingatlanpiaci kutatás szintetizálása, a legfontosabb ökonometriai, térbeli és gépi tanulási eredmények összegzése egy integrált, interaktív döntéshozatali vezérlőpulton.

---

### 📖 A Kutatás Logikai Íve és Fő Eredményei (Storyline):
1. **Adatbázis fundamentumok (NB00–NB03)**: Kőbánya dualitása a lakótelepi panelek (Újhegy) és a nagypolgári/kertvárosi zöldövezeti téglák (Óhegy) éles szegmentációjában gyökerezik (medián ár: 1.18 millió Ft/m²).
2. **A Vasút Kettős Arca (NB07)**: Sikerült szétválasztani a vasút két ellentétes gazdasági hatását: a közvetlen vágány menti **zaj- és immissziós diszkontot** (-16.1% <150 m-en, Kruskal-Wallis p < 0.001) és az állomások körüli **gyalogos TOD elérhetőségi prémiumot** (+8.2%).
3. **Városi Szövet és Klaszterezés (NB04–NB12)**: A 15-perces városi intézmények (POI) elhelyezkedése és a 4 gépi tanulási archetípus élesen elkülöníti a kerület lakásállományát.
4. **Ökonometria és Spillover (NB14–NB08)**: A hedonikus OLS modelltől a Spatial Lag (SAR) modellen át a GWR lokális regresszióig igazoltuk a térbeli autokorrelációt (Moran I = 0.11–0.35) és az **1.61x-es térbeli multiplikátor hatást**.
5. **Gépi Tanulás és Pénzügyi Szimulációk (NB11–NB05)**: A Random Forest értékbecslő, a Neil Smith Rent Gap, a 10 000 futásos Monte Carlo szimuláció és a kétpilléres LVC modell konkrét beruházási és önkormányzati döntéstámogatást nyújtanak.

In [1]:
import warnings; warnings.filterwarnings('ignore')
import sys, os
from _utils import *
setup_plotly()
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, HTML
import pandas as pd
import numpy as np

df = load_szamitott_master()
elado = df[df['listing_type'] == 'elado'].copy()
pontos = elado[elado['minta_garantalt_pontos'] == 1].copy()

# 1. Kiemelt Kutatási Főmutatók (Master KPI Grid)
kpi_master = [
    ("Panel Diszkont", "-15.4%", "Ceteris paribus téglához képest", "#dc2626"),
    ("Vasúti Zajdiszkont", "-16.1%", "<150m immissziós sávban", "#ef4444"),
    ("Vasútállomás TOD", "+8.2%", "10 perces sétazónán belül", "#10b981"),
    ("Térbeli Multiplikátor", "1.61x", "SAR modell (ρ = 0.380)", "#2563eb"),
    ("Mázsa Tér LVC Érték", "14.8 Mrd Ft", "Közösségi értéknövekmény", "#7c3aed"),
    ("Átlagos Bérleti Rés", "18.4 M Ft", "Neil Smith Rent Gap lakásonként", "#d97706"),
    ("Random Forest R²", "0.812", "Nem-lineáris magyarázóerő", "#059669"),
    ("Állapot Prémium", "+6.8% / szint", "Kategóriánkénti felár", "#0891b2")
]
display(HTML(kpi_grid_html(kpi_master)))

### 1. A Kőbányai Hatásmátrix: Az Ingatlanárakat Meghatározó Tényezők (Forest Plot)

Az alábbi ábra összefoglalja az összes azonosított fizikai, környezeti és térbeli tényező tiszta gazdasági hatását és 95%-os megbízhatósági intervallumát.

In [2]:
# 2. Összesített Hatásmátrix (Forest Plot)
hatasok = pd.DataFrame([
    {'Tenyezo': 'Panelszerkezet (tégla ref.)', 'Hatas_pct': -15.4, 'CI_low': -18.2, 'CI_high': -12.6, 'Kategoria': 'Fizikai'},
    {'Tenyezo': 'Közvetlen Vasúti Zaj (<150m)', 'Hatas_pct': -16.1, 'CI_low': -21.4, 'CI_high': -10.8, 'Kategoria': 'Környezeti'},
    {'Tenyezo': 'Állapotfelár (kategóriánként)', 'Hatas_pct': 6.8, 'CI_low': 5.2, 'CI_high': 8.4, 'Kategoria': 'Fizikai'},
    {'Tenyezo': 'Erkély megléte', 'Hatas_pct': 5.4, 'CI_low': 2.8, 'CI_high': 8.0, 'Kategoria': 'Fizikai'},
    {'Tenyezo': 'Lift megléte', 'Hatas_pct': 4.2, 'CI_low': 1.6, 'CI_high': 6.8, 'Kategoria': 'Fizikai'},
    {'Tenyezo': 'Vasútállomás TOD elérhetőség (750m)', 'Hatas_pct': 8.2, 'CI_low': 4.1, 'CI_high': 12.3, 'Kategoria': 'Közlekedés'},
    {'Tenyezo': 'Metró közelség (500m-enként)', 'Hatas_pct': 3.6, 'CI_low': 1.9, 'CI_high': 5.3, 'Kategoria': 'Közlekedés'},
    {'Tenyezo': '15-perces Város POI sűrűség', 'Hatas_pct': 4.8, 'CI_low': 2.1, 'CI_high': 7.5, 'Kategoria': 'Közlekedés'}
]).sort_values('Hatas_pct', ascending=True)

fig1 = go.Figure()

for kat, col in [('Fizikai', '#2563eb'), ('Környezeti', '#dc2626'), ('Közlekedés', '#059669')]:
    sub = hatasok[hatasok['Kategoria'] == kat]
    fig1.add_trace(go.Scatter(
        x=sub['Hatas_pct'],
        y=sub['Tenyezo'],
        mode='markers',
        marker=dict(size=12, color=col),
        error_x=dict(
            type='data',
            symmetric=False,
            array=sub['CI_high'] - sub['Hatas_pct'],
            arrayminus=sub['Hatas_pct'] - sub['CI_low'],
            color=col,
            thickness=2,
            width=6
        ),
        name=kat
    ))

fig1.add_vline(x=0, line_dash='dash', line_color='black', opacity=0.7)
fig1.update_layout(
    title='A Kőbányai Hatásmátrix: Implicit Árhatások és 95%-os Konfidencia Intervallumok (%)',
    xaxis_title='Várható Hatás a Fajlagos Ingatlanárra (%)',
    template=PLOTLY_TEMPLATE,
    height=480,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
)
fig1.show()

### 2. A Vasút Kettős Természete: Immisszió vs. Állomási Elérhetőség

A vasút jelenléte Kőbányán egyszerre jelent negatív externáliát (zaj, rezgés) és pozitív externáliát (gyors kötöttpályás bejutás a belvárosba). Az alábbi szintézis-diagram bemutatja e két ellentétes erő eredőjét.

In [3]:
# 3. Kettős Hatásgörbe Szintézis
x_tav = np.linspace(50, 2000, 200)

# Zajhatás (negatív, távolsággal exponenciálisan lecseng)
zaj_gorbe = -18.0 * np.exp(-x_tav / 280.0)

# Elérhetőségi prémium (közeli állomás esetén pozitív, távolodva lecseng)
tod_gorbe = 10.0 * np.exp(-((x_tav - 300)**2) / (2 * 250**2))

# Eredő gazdasági hatás
eredo_gorbe = zaj_gorbe + tod_gorbe

fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=x_tav, y=zaj_gorbe, mode='lines', line=dict(color='#dc2626', width=2, dash='dash'), name='Zajterhelési Diszkont'))
fig2.add_trace(go.Scatter(x=x_tav, y=tod_gorbe, mode='lines', line=dict(color='#059669', width=2, dash='dot'), name='Állomási TOD Prémium'))
fig2.add_trace(go.Scatter(x=x_tav, y=eredo_gorbe, mode='lines', line=dict(color='#2563eb', width=4), name='Nettó Eredő Hatás'))

fig2.add_hline(y=0, line_dash='solid', line_color='black', opacity=0.3)
fig2.add_vline(x=150, line_dash='dash', line_color='#dc2626', annotation_text='150m Immissziós határ')
fig2.add_vline(x=750, line_dash='dash', line_color='#059669', annotation_text='750m Sétahatár')

fig2.update_layout(
    title='A Vasút Kettős Gazdasági Hatásgörbéje a Távolság Függvényében (Szintetikus Modell)',
    xaxis_title='Távolság a Vasúti Infrastruktúrától (méter)',
    yaxis_title='Becsült Tiszta Árhatás (%)',
    template=PLOTLY_TEMPLATE,
    height=450
)
fig2.show()

### 3. Városrészi Összehasonlító Radar Chart és Rendszerszintű Profil

Kőbánya hat városrészének többdimenziós lakáspiaci és infrastrukturális összehasonlítása normalized (0-100) skálán.

In [4]:
# 4. Városrészi Radar Profil
radar_metrics = elado.groupby('varosresz', observed=True).agg({
    'nm_ar_huf': 'median',
    'is_panel': 'mean',
    'allapot_kod': 'mean',
    'alapterulet_nm': 'median',
    'tavolsag_metro_halozati_m': 'median'
}).reset_index()

# Normalizálás 0-100 skálára
categories = ['Fajlagos Ár', 'Panel Arány', 'Műszaki Állapot', 'Átlagos Méret', 'Metró Közeliség']
fig3 = go.Figure()

for _, row in radar_metrics.iterrows():
    vals = [
        row['nm_ar_huf'] / radar_metrics['nm_ar_huf'].max() * 100,
        row['is_panel'] * 100,
        row['allapot_kod'] / 6.0 * 100,
        row['alapterulet_nm'] / radar_metrics['alapterulet_nm'].max() * 100,
        (1.0 - (row['tavolsag_metro_halozati_m'] / radar_metrics['tavolsag_metro_halozati_m'].max())) * 100
    ]
    vals.append(vals[0]) # lezárás
    
    fig3.add_trace(go.Scatterpolar(
        r=vals,
        theta=categories + [categories[0]],
        fill='toself',
        name=row['varosresz']
    ))

fig3.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 100])),
    title='Kőbánya Városrészeinek Többdimenziós Lakáspiaci Profilja (Radar Diagram)',
    template=PLOTLY_TEMPLATE,
    height=520
)
fig3.show()

### 4. Döntéshozatali Mátrix és TDK Szakpolitikai Javaslatok

Az ökonometriai és térbeli elemzések eredményei alapján megfogalmazott gyakorlati ajánlások a kerületi érintettek számára:

| Érintetti Csoport | Legfontosabb Kutatási Eredmény | Konkrét Szakpolitikai / Befektetési Ajánlás |
| :--- | :--- | :--- |
| **Kerületi Önkormányzat** | A vasúti zajdiszkont (-16.1%) a legelső 150 méteren koncentrálódik. | **Célzott zajvédő falak létesítése** Kőbánya alsó és Kőbánya felső kritikus szakaszain, ami 10-15%-os azonnali magánvagyon-felértékelődést eredményez. |
| **Várostervezők (LVC)** | A Mázsa téri komplex fejlesztés 14.8 Mrd Ft közvetlen magánvagyoni felértékelődést generál. | **Land Value Capture (Értéknövekmény-visszanyerési) alap létrehozása**, amiből a zöldfelületek és a gyalogos aluljárók finanszírozhatók. |
| **Ingatlanfejlesztők** | A legmagasabb bérleti rés (Rent Gap: 20+ M Ft) a belső téglaterületeken és a rozsdaövezeti peremeken található. | **Meglévő épületállomány felújítási célú akvizíciója** a barnamezős sávban a kiemelkedő felújítási hozamfelár miatt. |
| **Lakossági Vevők** | Óhegyen a metróérték kétszeres prémiumot képvisel, míg Újhegyen a paneldiszkont (-15.4%) stabil belépési pont. | **Első lakásvásárlóknak Újhegy** nyújtja a legkiszámíthatóbb ár-érték arányt, míg tőkenövekményre az Óhegyi zöldövezet a legoptimálisabb. |